In [1]:
import os
import numpy as np
import pandas as pd

# -----------------------
# Settings
# -----------------------
LAMBDA = 0.94
H_INIT = 50                  # initialisation window for Sigma_0
SAVE_FROM = "2021-01-01"      # first date t you want to STORE Sigma_t for
IN_PATH = "../data/log_returns.csv"
OUT_PATH = "../models/ewma_cov_lambda094.csv"

# -----------------------
# Load returns
# -----------------------
rets = pd.read_csv(IN_PATH, parse_dates=["Date"]).set_index("Date").sort_index()
assets = rets.columns.tolist()
R = rets.to_numpy()
dates = rets.index
T, N = R.shape

if T <= H_INIT + 2:
    raise ValueError("Not enough data for the chosen H_INIT.")

# -----------------------
# 1) Initialise Sigma using first H_INIT returns
# -----------------------
Sigma = np.cov(R[:H_INIT].T, bias=False)  # sample covariance as a stable start

# -----------------------
# 2) Run EWMA recursion through the whole sample
#    IMPORTANT: store Sigma_t dated at t, computed using r_{t-1}
# -----------------------
sigmas = []
sigma_dates = []

# earliest t we can define Sigma_t for after init is t = dates[H_INIT]
# because Sigma_{dates[H_INIT]} uses r_{dates[H_INIT-1]} and Sigma_{t-1} exists
for k in range(H_INIT, T):
    # k corresponds to date t = dates[k]
    r_prev = R[k-1].reshape(N, 1)  # r_{t-1}
    Sigma = LAMBDA * Sigma + (1 - LAMBDA) * (r_prev @ r_prev.T)
    sigmas.append(Sigma.copy())
    sigma_dates.append(dates[k])

ewma_full = pd.Series(sigmas, index=pd.Index(sigma_dates, name="Date"))

# -----------------------
# 3) Keep only from SAVE_FROM (optional)
# -----------------------
ewma_full = ewma_full.loc[pd.to_datetime(SAVE_FROM):]

# -----------------------
# 4) Save as 36 unique elements (upper triangle incl diagonal)
# -----------------------
tri_i, tri_j = np.triu_indices(N)
colnames = [f"cov_{assets[i]}__{assets[j]}" for i, j in zip(tri_i, tri_j)]

ewma_df = pd.DataFrame(
    [M[tri_i, tri_j] for M in ewma_full.values],
    index=ewma_full.index,
    columns=colnames
)
ewma_df.index.name = "Date"

os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
ewma_df.to_csv(OUT_PATH)

print("Saved:", OUT_PATH)
print("EWMA shape:", ewma_df.shape)
print("EWMA date range:", ewma_df.index.min().date(), "->", ewma_df.index.max().date())
ewma_df.head()

Saved: ../models/ewma_cov_lambda094.csv
EWMA shape: (1255, 36)
EWMA date range: 2021-01-04 -> 2025-12-31


,cov_US_EQUITY__US_EQUITY,cov_US_EQUITY__INTL_EQUITY,cov_US_EQUITY__US_BONDS,cov_US_EQUITY__INTL_BONDS,cov_US_EQUITY__US_REIT,cov_US_EQUITY__INTL_REIT,cov_US_EQUITY__GOLD,cov_US_EQUITY__BTC,cov_INTL_EQUITY__INTL_EQUITY,cov_INTL_EQUITY__US_BONDS,...,cov_US_REIT__US_REIT,cov_US_REIT__INTL_REIT,cov_US_REIT__GOLD,cov_US_REIT__BTC,cov_INTL_REIT__INTL_REIT,cov_INTL_REIT__GOLD,cov_INTL_REIT__BTC,cov_GOLD__GOLD,cov_GOLD__BTC,cov_BTC__BTC
Date,,,,,,,,,,,,,,,,,,,,,
2021-01-04,0.000052,0.000042,-1.013893e-07,-0.000002,0.000053,0.000037,0.000017,0.000065,0.000074,-0.000003,...,0.000101,0.000051,0.000018,0.000087,0.000092,0.000017,4.921141e-05,0.000075,0.000044,0.002489
2021-01-05,0.000060,0.000036,9.614280e-07,-0.000001,0.000079,0.000043,-0.000003,-0.000020,0.000071,-0.000004,...,0.000166,0.000067,-0.000031,-0.000121,0.000092,0.000003,-9.993554e-06,0.000102,0.000176,0.002910
2021-01-06,0.000059,0.000038,5.012060e-07,-0.000002,0.000074,0.000047,-0.000002,0.000005,0.000076,-0.000004,...,0.000157,0.000063,-0.000029,-0.000114,0.000103,0.000005,5.292666e-05,0.000096,0.000173,0.002961
2021-01-07,0.000058,0.000040,-1.309565e-06,-0.000002,0.000069,0.000044,-0.000007,0.000034,0.000078,-0.000007,...,0.000147,0.000060,-0.000027,-0.000110,0.000097,0.000004,4.975106e-05,0.000105,0.000087,0.003167
2021-01-08,0.000068,0.000039,-2.133485e-06,-0.000002,0.000065,0.000031,-0.000010,0.000091,0.000073,-0.000007,...,0.000138,0.000056,-0.000025,-0.000103,0.000100,0.000007,-4.206621e-07,0.000100,0.000069,0.003246


In [2]:
proxy = pd.read_csv("../proxy/realized_cov_h21.csv", parse_dates=["Date"]).set_index("Date")
ewma  = pd.read_csv("../models/ewma_cov_lambda094.csv", parse_dates=["Date"]).set_index("Date")

common = proxy.index.intersection(ewma.index)
print("Common dates:", len(common), common.min().date(), "->", common.max().date())

Common dates: 1235 2021-01-04 -> 2025-12-02
